# Distributing work

An analysis often applies the same computation to many pieces of data: the observations of a
season, the detector batches of an observation, the frequency channels of a component separation.
The natural furax expression is a composite — a sum or a block operator — whose operands only
differ by the values of their arrays. The operands can be handed to the composite in two ways, as
a plain pytree of operators or as an [`Ensemble`][furax.core.Ensemble], and the choice decides
three things:

- **program size**: whether the computation is traced and compiled once, or once per operand;
- **peak memory**: how many operands' temporaries are alive at the same time;
- **placement**: which device holds and evaluates which operand.

This page explains when a pytree of operators is enough, when to switch to an ensemble, and walks
through a complete example that reads its observations from disk and distributes them over the
devices. The [Ensembles](ensembles.md) page is the reference for the machinery: the
representations, the engine and its output specification.

The examples of this page run with 64-bit precision enabled:

In [ ]:
import jax

jax.config.update('jax_enable_x64', True)

## A pytree of operators

The composites accept any pytree of operators — a list, a dict, a tuple. This is the route of
convenience, and the only one for operators of different types:

In [ ]:
import jax.numpy as jnp

from furax import DiagonalOperator
from furax.core import AdditionOperator

ops = [DiagonalOperator(jnp.full(4, 1.0 + i)) for i in range(8)]
total = AdditionOperator(ops)
total(jnp.ones(4))

By default the sum is evaluated as one expression, which lets XLA schedule the operands together,
each holding its temporaries: fine for small operands, not for eight TOD-sized ones. With
`sequential=True`, the operands are applied one at a time and the peak memory of an application is
set by the largest operand instead of all of them together.

A sequential sum also keeps the compiled program small: operands that only differ by the values of
their arrays — same class, same static fields, same array shapes and dtypes — are applied through
a single traced body, a loop that selects one operand's arrays at a time, so the program does not
grow with their number. [`BlockRowOperator`][furax.core.BlockRowOperator] applies the same
grouping to its blocks. This all happens on one device, one operand at a time: there is no
vectorized batching and no distribution.

## An ensemble of operators

The composites also accept an [`Ensemble`][furax.core.Ensemble] of operators: N members sharing
one structure, typically built member-wise by [`lift`][furax.core.lift] from the stacked arrays.
The composite then evaluates its blocks through the engine, and the ensemble's
[`Policy`][furax.core.Policy] makes the three decisions explicit: the body is traced once whatever
N, `Policy.batch` sets how many members run together (vectorized, bounding the memory peak to one
round's temporaries), and `Policy.axis` splits the members over the devices of a mesh axis.

In [ ]:
import jax.random as jr

from furax import IndexOperator
from furax.core import BlockColumnOperator, Policy, StackedEnsemble, lift

n_obs, n_samples, n_pixels = 16, 100, 32
key_x, key_p = jr.split(jr.key(0))
x = jr.normal(key_x, (n_pixels,), jnp.float32)
pixels = jr.randint(key_p, (n_obs, n_samples), 0, n_pixels)

structure = jax.ShapeDtypeStruct((n_pixels,), jnp.float32)
hs = lift(
    lambda m: IndexOperator(m['pixels'], in_structure=structure),
    StackedEnsemble(n_obs, {'pixels': pixels}, policy=Policy(batch=2)),
)
with jax.set_mesh(jax.make_mesh((jax.device_count(),), ('obs',))):
    H = BlockColumnOperator(hs.shard('obs'))
    y = H(x)  # the stacked TOD stays laid out over the devices
    normal = (H.T @ H).reduce()  # structural: no data is moved
    b = H.T(y)  # summed across the devices, held by all of them

The operator algebra is unchanged: transposition, composition and reduction apply member-wise,
structurally where possible, so building the normal operator never touches the stacked arrays.
[`Policy.from_budget`][furax.core.Policy.from_budget] derives the batch size from a device memory
budget, using XLA's memory analysis of one abstract member.

## Choosing

| Situation | Operands |
| --- | --- |
| a handful of blocks, possibly of different types | a pytree of operators |
| many same-structured operands, one device, one at a time | a pytree of operators (`sequential=True` for a sum), or a [`ListEnsemble`][furax.core.ListEnsemble] |
| members to run in vectorized rounds, or to split over devices | a [`StackedEnsemble`][furax.core.StackedEnsemble] |
| a dataset larger than the devices' memory | a [`HostEnsemble`][furax.core.HostEnsemble] |

The difference between the two routes is who decides. With a pytree of operators, the grouping of
same-structured operands is automatic but opportunistic, and everything runs on one device. With
an ensemble, the single traced body, the round size and the placement are declared, and the same
composite scales from one device to a mesh of processes.

## A complete example: mapmaking from observations on disk

The toy multi-observation problem of the [Ensembles](ensembles.md) page, taken one step closer to
reality: the observations now live on disk, each holding a TOD and its pointing, and the dataset
is walked without ever being materialized. Each observation is a pure pointing, yᵢ = x[pᵢ], and
the map solves the normal equations (Σᵢ Hᵢᵀ Hᵢ) x̂ = Σᵢ Hᵢᵀ yᵢ. The example runs on the available
devices; on a CPU, set `XLA_FLAGS=--xla_force_host_platform_device_count=8` before starting
Python to simulate eight.

### The dataset

Sixteen observations, each stored as one file with its TOD and its pixel indices — stand-ins for the
gigabyte-sized arrays of a real season — in a temporary directory:

In [ ]:
import tempfile
from pathlib import Path

import numpy as np

n_obs, n_dets, n_samples, n_pixels = 16, 3, 100, 32

rng = np.random.default_rng(0)
x_true = rng.normal(size=n_pixels).astype(np.float32)
data_dir = Path(tempfile.mkdtemp())
for i in range(n_obs):
    pixels = rng.integers(0, n_pixels, size=(n_dets, n_samples)).astype(np.int32)
    np.savez(data_dir / f'obs_{i:03d}.npz', tod=x_true[pixels], pixels=pixels)

### The pass over the data

A [`HostEnsemble`][furax.core.HostEnsemble] describes the dataset without loading it: a size, a
`read` callback returning one member, and a template giving its structure. The `read` callback
returns the member as numpy arrays with the structure of the template, against which their shapes
and dtypes are checked:

In [ ]:
from furax.core import HostEnsemble, map_reduce

structure = jax.ShapeDtypeStruct((n_pixels,), jnp.float32)
template = {
    'pixels': jax.ShapeDtypeStruct((n_dets, n_samples), jnp.int32),
    'tod': jax.ShapeDtypeStruct((n_dets, n_samples), jnp.float32),
}


def read(i: int) -> dict[str, np.ndarray]:
    with np.load(data_dir / f'obs_{i:03d}.npz') as f:
        return {'pixels': f['pixels'], 'tod': f['tod']}


def accumulate(member):
    h = IndexOperator(member['pixels'], in_structure=structure)
    return {
        'hits': h.T(jnp.ones_like(member['tod'])),
        'rhs': h.T(member['tod']),
        'pixels': member['pixels'],
    }


mesh = jax.make_mesh((jax.device_count(),), ('obs',))
with jax.set_mesh(mesh):
    observations = HostEnsemble(n_obs, read, template, policy=Policy(axis='obs'))
    out, _ = map_reduce(accumulate, observations, {'hits': 'sum', 'rhs': 'sum', 'pixels': 1})

The policy's axis distributes the members in position order: with D devices, device d evaluates
members d·N/D to (d+1)·N/D. Each round, every device receives one observation while the host reads
the next round in a thread, so the pass runs at the speed of the slower of the disk and the
devices. The output specification says what survives the pass, per leaf: `'sum'` leaves are summed
over the members, then across the devices, and land replicated; stride-1 leaves are stacked and
stay with the device that computed them:

In [ ]:
print(out['rhs'].sharding.spec)  # summed, held by every device
print(out['pixels'].sharding.spec)  # stacked, sharded in position order

At any instant, a device holds one observation's TOD and pointing — two with the prefetch — and
nothing else of the dataset: the TODs are never materialized together. Once the pass is over, no
TOD remains at all; only the summed maps and the stacked pointing survive.

### The solve

The stacked `pixels` output lands sharded in position order, which is exactly the layout of a
resident [`StackedEnsemble`][furax.core.StackedEnsemble] with the same policy: the output of the
pass over the data is the ensemble the solve then iterates on. [`lift`][furax.core.lift] turns it
into the ensemble of pointing operators, and the composite closes it into the normal operator:

In [ ]:
with jax.set_mesh(mesh):
    pointings = StackedEnsemble(
        n_obs, {'pixels': out['pixels']}, 1, policy=Policy(axis='obs', batch=2)
    )
    hs = lift(
        lambda m: IndexOperator(m['pixels'], in_structure=structure, out_structure=template['tod']),
        pointings,
    )
    H = BlockColumnOperator(hs)
    normal = (H.T @ H).reduce()
    x_hat, _ = jax.scipy.sparse.linalg.cg(normal, out['rhs'], tol=1e-6)

np.testing.assert_allclose(np.asarray(x_hat), x_true, atol=1e-5)

Two details are doing quiet work here. The lifted constructor receives `out_structure` explicitly,
so it does not compute anything from the indices: the lift is structural, and the ensemble of
operators reuses the sharded array in place — no gather, no copy. And `batch=2` makes every matrix
application of the conjugate gradient evaluate two members per device together, vectorized, while
the summed result is shared by all of them.

The dataset's TODs and pointings are never resident together: during the pass, each device holds
one observation's worth of both, and by the time the solve holds every pointing, the TODs are
gone, consumed into the right-hand side. The solve itself runs on the pointings and a few
map-sized vectors, nothing more. This is, at toy scale, the
architecture of the multi-observation mapmaker
([`MultiObservationMapMaker`][furax.mapmaking.MultiObservationMapMaker]): its pass over the data
is a host ensemble of observations whose stacked outputs seed the resident ensemble its solver
iterates on, with noise weighting, detector batches and gap treatment layered on top.